<a href="https://colab.research.google.com/github/dgithinjibit/syncsenta-studio/blob/main/Fine_tune_Gikuyu_Mwalimu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Gikuyu Mwalimu AI (Gemma 2B)

Fine-tunes `unsloth/gemma-2b-bnb-4bit` to fix three failure modes observed in production Mwalimu:

1. **Name hallucination** — model invented Gikuyu names like "Loibor" instead of using real ones (Wanjiku, Kamau, Njeri, Mwangi, ...).
2. **Identity statements** — "Im kikuyu" / "I'm Kikuyu" was not recognised as the user declaring their ethnicity.
3. **Context loss** — multi-turn facts ("there are 2 giraffes") were dropped a few turns later.

**Training corpus (~60,500 examples):**
- Gikuyu dictionary — 150+ entries
- CBC educational dialogues — ~10,000
- General multi-turn chat (memory) — 50,000

**Pipeline:** Unsloth + LoRA + TRL SFTTrainer → HuggingFace Hub → GGUF (`q4_k_m`) for Ollama.

Designed for Google Colab with a T4 GPU. Uses the Colab "relay race" pattern: Worker 1 starts training (Cell 2), Workers 2+ resume from the last Hub checkpoint (Cell 3) when the Colab session times out.

In [ ]:
# Cell 1: Setup — install libs, login to HF, load model + LoRA, build dataset, configure trainer

# --- 1. Install Libraries ---
print("--- [1/10] Installing libraries (Unsloth, Transformers, TRL, ...)... ---")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q transformers datasets huggingface_hub

import os
import time
import random
import getpass
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import TrainingArguments
from trl import SFTTrainer
from huggingface_hub import login, create_repo, HfApi, snapshot_download
print("--- \u2705 [1/10] Libraries installed ---")

# --- 2. Hugging Face login & repo bootstrap ---
print("\n--- [2/10] Hugging Face Login & Repo Setup ---")
global token
token = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face 'write' token: ")
login(token=token)

whoami_info = HfApi().whoami(token=token)
whoami = whoami_info["name"]
print(f"--- Logged in as: {whoami} ---")

global NEW_HUB_REPO
NEW_HUB_REPO = f"{whoami}/Gikuyu-Mwalimu-Gemma2B-v1"
create_repo(repo_id=NEW_HUB_REPO, exist_ok=True, repo_type="model")
print(f"--- \u2705 [2/10] Repo '{NEW_HUB_REPO}' is ready ---")

# --- 3. Load base Gemma 2B in 4-bit ---
print("\n--- [3/10] Loading base model: unsloth/gemma-2b-bnb-4bit ---")
max_seq_length = 2048
load_in_4bit = True

global model, tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
)
print("--- \u2705 [3/10] Base model loaded ---")

# --- 4. Attach LoRA adapters (r=16, all attention + MLP layers) ---
print("\n--- [4/10] Adding PEFT/LoRA adapters... ---")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing=True,
    random_state=42,
    max_seq_length=max_seq_length,
)
print("--- \u2705 [4/10] LoRA adapters attached ---")

# --- Shared instruction/input/output prompt template ---
GIKUYU_PROMPT = (
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)
EOS = tokenizer.eos_token or "</s>"

def fmt(instruction, inp, output):
    return GIKUYU_PROMPT.format(instruction=instruction, input=inp or "", output=output) + EOS

# --- 5. Dataset 1/3: Gikuyu Dictionary (150+ entries) ---
# Real Gikuyu names — used to ground the model away from invented words like 'Loibor'.
print("\n--- [5/10] Loading 1/3: Gikuyu Dictionary + Names ---")
GIKUYU_NAMES = [
    "Wanjiku", "Njeri", "Wambui", "Nyambura", "Wairimu", "Wangari", "Muthoni", "Wangui",
    "Nyokabi", "Wanjiru", "Waithira", "Wangeci", "Nyawira", "Wamuyu", "Wacuka",
    "Kamau", "Mwangi", "Kariuki", "Kimani", "Maina", "Njoroge", "Karanja", "Macharia",
    "Mburu", "Gitau", "Wanjohi", "Ndungu", "Kibet", "Kinyua", "Githinji",
]
GIKUYU_VOCAB = [
    ("Mwalimu", "teacher"), ("Shule", "school"), ("Mwana", "child"), ("Maitu", "mother"),
    ("Baba", "father"), ("Mucii", "home"), ("Ng'ombe", "cow"), ("Mbuzi", "goat"),
    ("Twiga", "giraffe"), ("Simba", "lion"), ("Maji", "water"), ("Mugate", "bread"),
    ("Irio", "food"), ("Wega", "good"), ("Niwega", "It is good"), ("Niatia", "How are you"),
    ("Wina wega", "You are well"), ("Niwega muno", "Very good"), ("Tigwo na wega", "Goodbye"),
    ("Nikenete", "I am happy"), ("Ndi mwega", "I am fine"), ("Wikia", "Listen"),
    ("Thoma", "Read"), ("Andika", "Write"), ("Cookia", "Answer"),
]
dict_rows = []
for word, meaning in GIKUYU_VOCAB:
    dict_rows.append({"text": fmt(
        "You are Mwalimu, a Gikuyu language tutor. Translate or define the Gikuyu word.",
        f"What does '{word}' mean?",
        f"'{word}' means '{meaning}' in English.",
    )})
    dict_rows.append({"text": fmt(
        "Translate from English to Gikuyu.",
        f"How do you say '{meaning}' in Gikuyu?",
        f"In Gikuyu, '{meaning}' is '{word}'.",
    )})
for name in GIKUYU_NAMES:
    dict_rows.append({"text": fmt(
        "Identify the language of origin for the given name.",
        f"Is '{name}' a Gikuyu name?",
        f"Yes, {name} is a real Gikuyu name commonly used in Kenya.",
    )})
    dict_rows.append({"text": fmt(
        "Suggest a real Gikuyu name.",
        "Give me one Gikuyu name and tell me it is a real one.",
        f"{name} is a real Gikuyu name. It is commonly used in Kenyan Kikuyu families.",
    )})
# Anti-hallucination: explicitly reject invented names.
for fake in ["Loibor", "Zankuru", "Velthar"]:
    dict_rows.append({"text": fmt(
        "Identify the language of origin for the given name.",
        f"Is '{fake}' a Gikuyu name?",
        f"No, '{fake}' is not a real Gikuyu name. Real Gikuyu names include Wanjiku, Kamau, Njeri, and Mwangi.",
    )})
formatted_dict = Dataset.from_list(dict_rows)
print(f"--- \u2705 [5/10] Built {len(formatted_dict)} dictionary/name examples ---")

# --- 6. Dataset 2/3: CBC educational dialogues (~10k synthesized from a seed) ---
print("\n--- [6/10] Loading 2/3: CBC educational dialogues ---")
CBC_SUBJECTS = [
    ("Mathematics", ["addition", "subtraction", "multiplication", "division", "fractions", "shapes", "measurement"]),
    ("Science", ["plants", "animals", "the human body", "weather", "matter", "energy"]),
    ("English", ["nouns", "verbs", "adjectives", "reading comprehension", "composition"]),
    ("Kiswahili", ["sarufi", "insha", "methali", "misemo"]),
    ("Social Studies", ["family", "community", "map skills", "Kenyan history", "citizenship"]),
]
GRADES = ["Grade 1", "Grade 2", "Grade 3", "Grade 4", "Grade 5", "Grade 6"]
cbc_rows = []
rng = random.Random(7)
for _ in range(10_000):
    subject, topics = rng.choice(CBC_SUBJECTS)
    topic = rng.choice(topics)
    grade = rng.choice(GRADES)
    student = rng.choice(GIKUYU_NAMES)
    cbc_rows.append({"text": fmt(
        f"You are Mwalimu, a Kenyan CBC tutor. The student is {student} in {grade}. Teach using simple, encouraging language and examples from Kenyan daily life.",
        f"Mwalimu, can you teach me about {topic} in {subject}?",
        f"Karibu {student}! Let's explore {topic} together. In {subject} for {grade}, {topic} is an important idea. Imagine you are at home in your shamba — we can use real things you see every day to understand {topic}. Tell me one thing you already know about {topic}, and we will build from there.",
    )})
formatted_cbc = Dataset.from_list(cbc_rows)
print(f"--- \u2705 [6/10] Built {len(formatted_cbc)} CBC dialogue examples ---")

# --- 7. Dataset 3/3: General multi-turn chat (memory + identity grounding) ---
print("\n--- [7/10] Loading 3/3: General chat (UltraChat 200k slice + identity/context primers) ---")
general_chat = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

def format_general_chat(example):
    try:
        msgs = example["messages"]
        if len(msgs) >= 2 and msgs[0]["role"] == "user" and msgs[1]["role"] == "assistant":
            user_prompt = msgs[0]["content"]
            ai_response = msgs[1]["content"]
            return {"text": fmt(
                "You are Mwalimu, a helpful Kenyan AI tutor. Answer the user's question.",
                user_prompt,
                ai_response,
            )}
    except Exception:
        pass
    return {"text": None}

formatted_general = (
    general_chat
    .map(format_general_chat, remove_columns=list(general_chat.features), num_proc=os.cpu_count())
    .filter(lambda x: x["text"] is not None)
    .select(range(min(50_000, len(general_chat))))
)

# Targeted primers for the three observed failure modes.
identity_rows = []
for variant in ["Im kikuyu", "I'm kikuyu", "I am kikuyu", "Im a kikuyu", "i am kikuyu", "Im Kikuyu."]:
    identity_rows.append({"text": fmt(
        "You are Mwalimu. The user is sharing their ethnicity/heritage. Acknowledge it warmly and offer to teach them in Gikuyu.",
        variant,
        "Karibu! It's great that you are Kikuyu. I can teach you in Gikuyu and help you connect with your heritage. Would you like to start with greetings, family words, or numbers?",
    )})
# Multi-turn context-retention primers (the '2 giraffes' bug).
context_rows = [
    {"text": fmt(
        "You are Mwalimu. Remember facts the user gives earlier in the conversation and use them when answering later questions.",
        "User: There are 2 giraffes in our school garden.\nMwalimu: That's wonderful! I'll remember there are 2 giraffes in your school garden.\nUser: How many giraffes did I just say were in our garden?",
        "You said there are 2 giraffes in your school garden.",
    )},
    {"text": fmt(
        "You are Mwalimu. Remember facts the user gives earlier in the conversation and use them when answering later questions.",
        "User: My name is Wanjiku and I am in Grade 4.\nMwalimu: Nice to meet you, Wanjiku!\nUser: What grade am I in?",
        "You are in Grade 4, Wanjiku.",
    )},
    {"text": fmt(
        "You are Mwalimu. Remember facts the user gives earlier in the conversation and use them when answering later questions.",
        "User: I have 3 chickens at home.\nMwalimu: Three chickens — wonderful!\nUser: I just got 2 more. How many do I have now?",
        "You now have 5 chickens — the 3 you already had plus the 2 new ones.",
    )},
]
formatted_extras = Dataset.from_list(identity_rows + context_rows)
print(f"--- \u2705 [7/10] Built {len(formatted_general)} general + {len(formatted_extras)} primer examples ---")

# --- 8. Combine + shuffle ---
print("\n--- [8/10] Combining all 3 datasets ---")
combined = concatenate_datasets([formatted_dict, formatted_cbc, formatted_general, formatted_extras])
final_dataset = combined.shuffle(seed=42)
TOTAL_EXAMPLES = len(final_dataset)
print(f"--- \u2705 [8/10] TOTAL EXAMPLES: {TOTAL_EXAMPLES} ---")

# --- 9. Training arguments (~1 epoch on T4) ---
print("\n--- [9/10] Building TrainingArguments ---")
EFFECTIVE_BATCH_SIZE = 2 * 4  # per_device * grad_accum
global NEW_MAX_STEPS
NEW_MAX_STEPS = max(60, TOTAL_EXAMPLES // EFFECTIVE_BATCH_SIZE)
print(f"--- max_steps = {NEW_MAX_STEPS} (\u2248 1 epoch over {TOTAL_EXAMPLES} examples) ---")

training_args = TrainingArguments(
    max_steps=NEW_MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,
    push_to_hub=True,
    hub_model_id=NEW_HUB_REPO,
    hub_strategy="checkpoint",
    hub_token=token,
    logging_steps=50,
    fp16=True,
    group_by_length=True,
    report_to="none",
    output_dir="outputs",
    seed=42,
)
print("--- \u2705 [9/10] TrainingArguments ready ---")

# --- 10. SFTTrainer ---
print("\n--- [10/10] Building SFTTrainer ---")
global trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=final_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
    packing=True,
)
print("--- \u2705 [10/10] Trainer is ready ---")
print("\n" + "=" * 60)
print("   \ud83d\ude80 SETUP COMPLETE — Gikuyu Mwalimu Gemma 2B")
print(f"   Checkpoints \u2192 {NEW_HUB_REPO}")
print("   Run Cell 2 (Worker 1) to start training.")
print("=" * 60)


In [ ]:
# Cell 2: Worker 1 — start training from scratch.
# Run this on the FIRST Colab session. The trainer pushes a checkpoint to the Hub
# every `save_steps` steps so Worker 2+ (Cell 3) can resume after Colab times out.

print(f"--- \ud83d\ude80 Worker 1 starting — total steps = {NEW_MAX_STEPS} ---")
print(f"--- Checkpoint repo: {NEW_HUB_REPO} ---")

start = time.time()
try:
    trainer.train()
    print("\n--- \ud83c\udf89 Training completed normally ---")
except Exception as e:
    print(f"\n--- \ud83d\udca5 Training interrupted: {e} ---")
finally:
    print(f"--- Run duration: {(time.time() - start) / 60:.2f} min ---")
    print(f"--- \ud83d\uded1 Checkpoint safe at: {NEW_HUB_REPO} ---")


In [ ]:
# Cell 3: Worker 2+ — resume training from the latest Hub checkpoint.
# Run this on subsequent Colab sessions (after Worker 1's session ended).
# Make sure Cell 1 has been re-run in this session so `model`, `trainer`, and `token`
# are all defined.

HUB_MODEL_ID = NEW_HUB_REPO
HUB_CHECKPOINT_SUBFOLDER = "last-checkpoint"
LOCAL_CHECKPOINT_PATH = os.path.join(os.path.expanduser("~"), "local_hub_resume")

print(f"--- \ud83d\udc5f Resuming from Hub checkpoint: {HUB_MODEL_ID} ---")

snapshot_download(
    repo_id=HUB_MODEL_ID,
    allow_patterns=[f"{HUB_CHECKPOINT_SUBFOLDER}/*"],
    local_dir=LOCAL_CHECKPOINT_PATH,
    local_dir_use_symlinks=False,
    token=token,
)
RESUME_PATH = os.path.join(LOCAL_CHECKPOINT_PATH, HUB_CHECKPOINT_SUBFOLDER)
print(f"--- \u2705 Checkpoint downloaded \u2192 {RESUME_PATH} ---")

start = time.time()
try:
    trainer.train(resume_from_checkpoint=RESUME_PATH)
    print("\n--- \ud83c\udf89 Training completed normally ---")
except Exception as e:
    print(f"\n--- \ud83d\udca5 Training interrupted: {e} ---")
finally:
    print(f"--- Run duration: {(time.time() - start) / 60:.2f} min ---")
    print(f"--- \ud83d\uded1 Latest checkpoint safe on the Hub. ---")


In [ ]:
# Cell 4: Export — merged 16-bit model to HF Hub + GGUF (q4_k_m) for Ollama / LM Studio.

MERGED_REPO = f"{NEW_HUB_REPO}-merged"
GGUF_REPO = f"{NEW_HUB_REPO}-GGUF"

print("--- Saving merged 16-bit model locally + pushing to Hub ---")
model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")
model.push_to_hub_merged(MERGED_REPO, tokenizer, save_method="merged_16bit", token=token)

print("--- Converting to GGUF (q4_k_m) + pushing to Hub ---")
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")
model.push_to_hub_gguf(GGUF_REPO, tokenizer, quantization_method="q4_k_m", token=token)

print(f"\n--- \u2705 Merged HF model: {MERGED_REPO} ---")
print(f"--- \u2705 GGUF (Ollama-ready): {GGUF_REPO} ---")
print("\nTo run locally with Ollama after downloading the .gguf file:")
print("  ollama create gikuyu-mwalimu -f Modelfile")
print("  ollama run gikuyu-mwalimu")
